In [1]:
# Imports

import os
import glob
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from tqdm.auto import tqdm
import open_clip
from sklearn.model_selection import train_test_split

In [2]:
# Device & Seed

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

device = torch.device(
    "mps" if torch.backends.mps.is_available()
    else "cuda" if torch.cuda.is_available()
    else "cpu"
)
print("device:", device)
# print(torch.cuda.is_available())
# print(torch.cuda.get_device_name(0))

device: cuda


In [3]:
# Configs

DATA_DIR = "./data"
TRAIN_DIR = os.path.join(DATA_DIR, "train")
TEST_DIR = os.path.join(DATA_DIR, "test")
CKPT_DIR = "./checkpoints/"
os.makedirs(CKPT_DIR, exist_ok=True)

NUM_CLASSES = 100
BATCH_SIZE = 64 # pick 32 if mips / mbp m1 pro
# Larger batch size makes gradient estimate more accurate (averaged over more images)
NUM_WORKERS = 0
EPOCHS_HEAD = 8 # pick 5 if mips / mbp m1 pro
EPOCHS_FINETUNE = 40 # pick 20 if mips / mbp m1 pro
CKPT_PATH = os.path.join(CKPT_DIR, "best_clip.pt")

# ViT-B-32 - Vision Transformer, 32x32 patches, pretrained by OpenAI
CLIP_MODEL = "ViT-B-32"
CLIP_PRETRAINED = "openai"

# https://github.com/mlfoundations/open_clip
# No IMG_SIZE, CLIP handles its own resizing internally
# No IMAGENET_MEAN/IMAGENET_STD - CLIP has its own normalization built in

# 400M image-text pairs - https://arxiv.org/abs/2103.00020

In [4]:
# Load CLIP model

model, _, preprocess = open_clip.create_model_and_transforms(
    CLIP_MODEL, pretrained=CLIP_PRETRAINED
)
# https://pytorch.org/docs/stable/generated/torch.nn.Module.html#torch.nn.Module.to
model = model.to(device)

print(f"Model: {CLIP_MODEL} pretrained on {CLIP_PRETRAINED}")
print(f"Total params: {sum(p.numel() for p in model.parameters()):,}")      

C:\venvs\ml312\Lib\site-packages\open_clip\factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(


Model: ViT-B-32 pretrained on openai
Total params: 151,277,313


In [5]:
# Dataset classes

class LabeledDataset(Dataset):
    def __init__(self, samples, transform=None):
        self.samples = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, label

class TestDataset(Dataset):
    def __init__(self, test_dir, transform=None):
        self.paths = sorted(
            glob.glob(os.path.join(test_dir, "*.jpg")),
            key=lambda x: int(os.path.splitext(os.path.basename(x))[0])
        )
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        path = self.paths[idx]
        img = Image.open(path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, os.path.basename(path)

In [6]:
# Build sample list and split
# Identical to the EfficientNet notebook.
# Should print Train: 863, Val: 216 again.

all_paths, all_labels = [], []
for class_id in range(NUM_CLASSES):
  class_dir = os.path.join(TRAIN_DIR, str(class_id))
  for fname in sorted(os.listdir(class_dir)):
      if fname.endswith(".jpg"):
          all_paths.append(os.path.join(class_dir, fname))
          all_labels.append(class_id)

tr_paths, vl_paths, tr_labels, vl_labels = train_test_split(
  all_paths, all_labels, test_size=0.2, stratify=all_labels, random_state=42
)

train_samples = list(zip(tr_paths, tr_labels))
val_samples = list(zip(vl_paths, vl_labels))

print(f"Train: {len(train_samples)}, Val: {len(val_samples)}")

Train: 863, Val: 216


In [7]:
# Dataloaders

train_ds = LabeledDataset(train_samples, transform=preprocess)
val_ds   = LabeledDataset(val_samples, transform=preprocess)
test_ds  = TestDataset(TEST_DIR, transform=preprocess)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

print(f"Batches — train: {len(train_loader)}, val: {len(val_loader)}, test: {len(test_loader)}")

Batches — train: 14, val: 4, test: 17


In [8]:
# Classification head

class CLIPClassifier(nn.Module):
  def __init__(self, clip_model, num_classes=NUM_CLASSES, freeze_backbone=True):
      super().__init__()
      self.clip = clip_model
      self.head = nn.Sequential(
          nn.Dropout(0.3),
          nn.Linear(512, num_classes),
      )
      if freeze_backbone:
          for param in self.clip.parameters():
              param.requires_grad = False

  def forward(self, x):
      features = self.clip.encode_image(x)
      features = features.float()
      return self.head(features)

classifier = CLIPClassifier(model, freeze_backbone=True).to(device)
trainable = sum(p.numel() for p in classifier.parameters() if p.requires_grad)
total     = sum(p.numel() for p in classifier.parameters())
print(f"Trainable: {trainable:,} / {total:,} params")

# CLIP's encode_image(x) vision encoder extracts 512-dim feature vector per image, replacing EfficientNet's model.features
# CLIP internally uses float16 for speed, convert to float32 for stable training
# Linear goes from 512 (CLIP's output size for ViT-B-32) to 100 classes
# 0.3 Dropout instead of 0.4 from EfficientNet, CLIP features are more generalizable, less regularization needed.

Trainable: 51,300 / 151,328,613 params


In [9]:
# Trainable: 51,300 / 151,328,613 params
# Compare that to EfficientNet phase 1 which had 1.69M trainable
# Surprisingly, our head is actually much smaller
# CLIP's 512-dim output is more compact than EfficientNet's 1536

In [10]:
# Loss function and training functions

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

def train_one_epoch(model, loader, optimizer, scheduler=None):
  model.train()
  total_loss, total_correct, n = 0.0, 0, 0
  for imgs, labels in tqdm(loader, leave=False):
      imgs, labels = imgs.to(device), labels.to(device)
      optimizer.zero_grad()
      out = model(imgs)
      loss = criterion(out, labels)
      loss.backward()
      optimizer.step()
      total_loss    += loss.item() * imgs.size(0)
      total_correct += (out.argmax(1) == labels).sum().item()
      n             += imgs.size(0)
  if scheduler:
      scheduler.step()
  return total_loss / n, total_correct / n


@torch.no_grad()
def evaluate(model, loader):
  model.eval()
  total_loss, total_correct, n = 0.0, 0, 0
  for imgs, labels in loader:
      imgs, labels = imgs.to(device), labels.to(device)
      out  = model(imgs)
      loss = criterion(out, labels)
      total_loss    += loss.item() * imgs.size(0)
      total_correct += (out.argmax(1) == labels).sum().item()
      n             += imgs.size(0)
  return total_loss / n, total_correct / n

# Identical to EfficientNet notebook. Runs forward pass, computes loss, backprops.

In [11]:
# Phase 1 training (head only)
optimizer_head = optim.AdamW(
  filter(lambda p: p.requires_grad, classifier.parameters()),
  lr=1e-3, weight_decay=1e-4
)
scheduler_head = optim.lr_scheduler.CosineAnnealingLR(optimizer_head, T_max=EPOCHS_HEAD)

history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
best_val_acc = 0.0

print("=== Phase 1: Head only ===")
for epoch in range(EPOCHS_HEAD):
  tr_loss, tr_acc = train_one_epoch(classifier, train_loader, optimizer_head, scheduler_head)
  vl_loss, vl_acc = evaluate(classifier, val_loader)
  for k, v in zip(["train_loss","train_acc","val_loss","val_acc"],
                  [tr_loss, tr_acc, vl_loss, vl_acc]):
      history[k].append(v)
  if vl_acc > best_val_acc:
      best_val_acc = vl_acc
      torch.save({"model_state_dict": classifier.state_dict(), "epoch": epoch, "val_acc": vl_acc}, CKPT_PATH)
  print(f"  [{epoch+1}/{EPOCHS_HEAD}] train {tr_acc:.4f} | val {vl_acc:.4f}")

print(f"\nBest so far: {best_val_acc:.4f}")

=== Phase 1: Head only ===


  0%|          | 0/14 [00:00<?, ?it/s]

  [1/8] train 0.0834 | val 0.2407


  0%|          | 0/14 [00:00<?, ?it/s]

  [2/8] train 0.2839 | val 0.3519


...

  [6/8] train 0.6049 | val 0.5602


  0%|          | 0/14 [00:00<?, ?it/s]

  [7/8] train 0.6385 | val 0.5741


  0%|          | 0/14 [00:00<?, ?it/s]

  [8/8] train 0.6512 | val 0.5741

Best so far: 0.5741


In [12]:
# Phase 2 training (full fine-tuning)

# PROGRESSIVE FREEZING (3 at a time, 12 blocks in CLIP - see last cell)
EPOCHS_PER_STAGE = 10

unfreeze_schedule = [
  [11, 10, 9],
  [8, 7, 6],
  [5, 4, 3],
  [2, 1, 0],
]

for stage, blocks in enumerate(unfreeze_schedule):
  for block_idx in blocks:
      for param in classifier.clip.visual.transformer.resblocks[block_idx].parameters():
          param.requires_grad = True
  if stage == 0:
      for param in classifier.clip.visual.ln_post.parameters():
          param.requires_grad = True
      classifier.clip.visual.proj.requires_grad = True

  trainable = sum(p.numel() for p in classifier.parameters() if p.requires_grad)
  print(f"\n=== Stage {stage+1}: blocks {blocks} | trainable: {trainable:,} ===")

  optimizer_stage = optim.AdamW([
      {"params": [p for p in classifier.clip.parameters() if p.requires_grad], "lr": 1e-6},
      {"params": classifier.head.parameters(), "lr": 1e-4},
  ], weight_decay=1e-4)
  scheduler_stage = optim.lr_scheduler.CosineAnnealingLR(optimizer_stage, T_max=EPOCHS_PER_STAGE)

  for epoch in range(EPOCHS_PER_STAGE):
      tr_loss, tr_acc = train_one_epoch(classifier, train_loader, optimizer_stage, scheduler_stage)
      vl_loss, vl_acc = evaluate(classifier, val_loader)
      for k, v in zip(["train_loss","train_acc","val_loss","val_acc"],
                      [tr_loss, tr_acc, vl_loss, vl_acc]):
          history[k].append(v)
      if vl_acc > best_val_acc:
          best_val_acc = vl_acc
          torch.save({"model_state_dict": classifier.state_dict(),
                      "epoch": EPOCHS_HEAD + stage * EPOCHS_PER_STAGE + epoch,
                      "val_acc": vl_acc}, CKPT_PATH)
      print(f"  [{epoch+1}/{EPOCHS_PER_STAGE}] train {tr_acc:.4f} | val {vl_acc:.4f}")

print(f"\nBest val acc: {best_val_acc:.4f}")

# NO PROGRESSIVE FREEZING
# for param in classifier.parameters():
#     param.requires_grad = True

# optimizer_ft = optim.AdamW([
#     {"params": classifier.clip.parameters(),  "lr": 1e-6},
#     {"params": classifier.head.parameters(),  "lr": 1e-4},
# ], weight_decay=1e-4)
# scheduler_ft = optim.lr_scheduler.CosineAnnealingLR(optimizer_ft, T_max=EPOCHS_FINETUNE)

# print("=== Phase 2: Full fine-tuning ===")
# for epoch in range(EPOCHS_FINETUNE):
#     tr_loss, tr_acc = train_one_epoch(classifier, train_loader, optimizer_ft, scheduler_ft)
#     vl_loss, vl_acc = evaluate(classifier, val_loader)
#     for k, v in zip(["train_loss","train_acc","val_loss","val_acc"],
#                     [tr_loss, tr_acc, vl_loss, vl_acc]):
#         history[k].append(v)
#     if vl_acc > best_val_acc:
#         best_val_acc = vl_acc
#         torch.save({"model_state_dict": classifier.state_dict(),
#                     "epoch": EPOCHS_HEAD + epoch, "val_acc": vl_acc}, CKPT_PATH)
#         print(f"  [{epoch+1}/{EPOCHS_FINETUNE}] train {tr_acc:.4f} | val {vl_acc:.4f}")

# print(f"\nBest val acc: {best_val_acc:.4f}")


=== Stage 1: blocks [11, 10, 9] | trainable: 21,709,668 ===


  0%|          | 0/14 [00:00<?, ?it/s]

  [1/10] train 0.6315 | val 0.5880


  0%|          | 0/14 [00:00<?, ?it/s]

  [2/10] train 0.6802 | val 0.5926


...

  [8/10] train 0.9803 | val 0.7963


  0%|          | 0/14 [00:00<?, ?it/s]

  [9/10] train 0.9815 | val 0.7963


  0%|          | 0/14 [00:00<?, ?it/s]

  [10/10] train 0.9791 | val 0.7963

Best val acc: 0.7963


In [13]:
# Submission

checkpoint = torch.load(CKPT_PATH, map_location=device)
classifier.load_state_dict(checkpoint["model_state_dict"])
print(f"Loaded checkpoint: epoch {checkpoint['epoch']}, val_acc {checkpoint['val_acc']:.4f}")

classifier.eval()
ids, preds = [], []
with torch.no_grad():
  for imgs, names in tqdm(test_loader):
      imgs = imgs.to(device)
      out  = classifier(imgs)
      ids.extend(names)
      preds.extend(out.argmax(1).cpu().tolist())

sub = pd.DataFrame({"ID": ids, "Label": preds})
sub.to_csv("submission_clip.csv", index=False)
print(sub.head(10))
print(f"Saved submission_clip.csv ({len(sub)} rows)")

Loaded checkpoint: epoch 44, val_acc 0.7963


  0%|          | 0/17 [00:00<?, ?it/s]

      ID  Label
0  0.jpg     62
1  1.jpg     43
2  2.jpg     38
3  3.jpg     51
4  4.jpg     37
5  5.jpg     89
6  6.jpg      3
7  7.jpg     49
8  8.jpg     72
9  9.jpg     65
Saved submission_clip.csv (1036 rows)


In [14]:
# Temp script to determine blocks in the visual transformer

# for name, param in classifier.clip.named_parameters():
#     print(name)

# CLIP's visual transformer has 12 blocks (visual.transformer.resblocks 0 through 11).
# Going to try to unfreeze 3 blocks at a time, from top to bottom.

positional_embedding
text_projection
logit_scale
visual.class_embedding
visual.positional_embedding
visual.proj
visual.conv1.weight
visual.ln_pre.weight
visual.ln_pre.bias
visual.transformer.resblocks.0.ln_1.weight
visual.transformer.resblocks.0.ln_1.bias
visual.transformer.resblocks.0.attn.in_proj_weight
visual.transformer.resblocks.0.attn.in_proj_bias
visual.transformer.resblocks.0.attn.out_proj.weight
visual.transformer.resblocks.0.attn.out_proj.bias
visual.transformer.resblocks.0.ln_2.weight
visual.transformer.resblocks.0.ln_2.bias
visual.transformer.resblocks.0.mlp.c_fc.weight
visual.transformer.resblocks.0.mlp.c_fc.bias
visual.transformer.resblocks.0.mlp.c_proj.weight
visual.transformer.resblocks.0.mlp.c_proj.bias
visual.transformer.resblocks.1.ln_1.weight
visual.transformer.resblocks.1.ln_1.bias
visual.transformer.resblocks.1.attn.in_proj_weight
visual.transformer.resblocks.1.attn.in_proj_bias
visual.transformer.resblocks.1.attn.out_proj.weight
visual.transformer.resblocks.1.att